In [1]:
import os
import urllib.request

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, classification_report,
)
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

SEED = 42
set_seed(SEED)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [2]:
MODEL_NAME = "klue/bert-base"
DATA_URL   = "https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/naver_shopping.txt"
DATA_PATH  = "naver_shopping.txt"
MAX_LEN    = 128
OUTPUT_DIR = "./shopping_sentiment_out"

In [3]:
if not os.path.exists(DATA_PATH):
    print("데이터 다운로드 중...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
print("데이터 준비 완료")

데이터 준비 완료


In [4]:
df = pd.read_csv(DATA_PATH, sep="\t", names=["rating", "review"])

df["label"] = (df["rating"] >= 4).astype(int)

df = df.dropna(subset=["review"])
df = df[df["review"].str.strip().str.len() > 0].reset_index(drop=True)

print(df.shape)
print(df["label"].value_counts())

(200000, 3)
label
0    100037
1     99963
Name: count, dtype: int64


In [5]:
trainval_df, test_df = train_test_split(
    df, test_size=0.1, stratify=df["label"], random_state=SEED
)
train_df, val_df = train_test_split(
    trainval_df, test_size=0.1, stratify=trainval_df["label"], random_state=SEED
)
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

train=162000  val=18000  test=20000


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(frame):
    return Dataset.from_pandas(frame[["review", "label"]], preserve_index=False)

def tokenize(batch):
    return tokenizer(batch["review"], truncation=True, max_length=MAX_LEN)

train_ds = to_ds(train_df).map(tokenize, batched=True)
val_ds   = to_ds(val_df).map(tokenize, batched=True)
test_ds  = to_ds(test_df).map(tokenize, batched=True)

collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/162000 [00:00<?, ? examples/s]

Map:   0%|          | 0/18000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":  accuracy_score(labels, preds),
        "f1":        f1_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall":    recall_score(labels, preds),
    }

In [9]:
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    bf16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    dataloader_num_workers=4,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [11]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,   # tokenizer= 에서 이걸로 변경
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.171676,0.166510,0.940389,0.940505,0.938371,0.942648
2,0.139519,0.168946,0.942278,0.942549,0.937830,0.947316
3,0.112119,0.181811,0.941222,0.941669,0.934252,0.949205


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7596, training_loss=0.1608824009265066, metrics={'train_runtime': 251.9479, 'train_samples_per_second': 1928.97, 'train_steps_per_second': 30.149, 'total_flos': 1.880181734937984e+16, 'train_loss': 0.1608824009265066, 'epoch': 3.0})

In [12]:
print("===== TEST 성능 =====")
test_metrics = trainer.evaluate(test_ds)
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

pred_out = trainer.predict(test_ds)
y_pred = np.argmax(pred_out.predictions, axis=-1)
y_true = pred_out.label_ids
print(classification_report(y_true, y_pred, target_names=["부정", "긍정"], digits=4))

===== TEST 성능 =====


Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.112119,0.165509,3,0.941350,0.941476,0.939086,0.943878


eval_loss: 0.1655
eval_accuracy: 0.9414
eval_f1: 0.9415
eval_precision: 0.9391
eval_recall: 0.9439


              precision    recall  f1-score   support

          부정     0.9436    0.9388    0.9412     10004
          긍정     0.9391    0.9439    0.9415      9996

    accuracy                         0.9414     20000
   macro avg     0.9414    0.9414    0.9413     20000
weighted avg     0.9414    0.9414    0.9413     20000



In [ ]:
save_path = os.path.join(OUTPUT_DIR, "best")
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"모델 저장 완료 → {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
def predict(texts):
    model.eval()
    enc = tokenizer(
        texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        logits = model(**enc).logits
    probs = torch.softmax(logits, dim=-1)
    preds = probs.argmax(dim=-1)
    label_map = {0: "부정", 1: "긍정"}
    for t, p, pr in zip(texts, preds, probs):
        print(f"[{label_map[p.item()]}] (확률 {pr[p].item():.3f})  {t}")

predict([
    "배송도 빠르고 품질도 최고예요 재구매 의사 있습니다",
    "포장이 엉망이고 제품에 흠집이 가득했어요 환불합니다",
    "가격대비 그냥 무난한 듯",
])